# [Experimental] Generate CTGAN Synthetic Data from Fraud Dataset

Uses a Conditional Tabular GAN (CTGAN) trained on the Kaggle fraud detection
dataset (`brian_gen_ai.xgb_scaling.kaggle_fraud_detection`) to generate
statistically faithful synthetic data at scale.

**This is experimental.** CTGAN training is slow, generated data quality varies,
and it may not scale well beyond 10M rows.

**Tables produced:** `ctgan_fraud_10k`, `ctgan_fraud_1m`, `ctgan_fraud_10m`, etc.

**Parameters:**
- `target_rows`: Number of synthetic rows to generate
- `table_suffix`: Suffix for output table name
- `n_noise_features`: Number of additional noise features to pad to target width
- `ctgan_epochs`: Training epochs for CTGAN (default 300)
- `ctgan_batch_size`: Batch size for CTGAN training (default 500)

In [ ]:
%pip install sdv ctgan
%restart_python

## Setup

In [ ]:
dbutils.widgets.text("target_rows", "10000", "Target Rows")
dbutils.widgets.text("table_suffix", "10k", "Table Suffix")
dbutils.widgets.text("n_noise_features", "240", "Noise Features to Add")
dbutils.widgets.text("ctgan_epochs", "300", "CTGAN Epochs")
dbutils.widgets.text("ctgan_batch_size", "500", "CTGAN Batch Size")
dbutils.widgets.text("catalog", "brian_gen_ai", "Catalog")
dbutils.widgets.text("schema", "xgb_scaling", "Schema")
dbutils.widgets.text("seed", "42", "Random Seed")

target_rows = int(dbutils.widgets.get("target_rows"))
table_suffix = dbutils.widgets.get("table_suffix")
n_noise_features = int(dbutils.widgets.get("n_noise_features"))
ctgan_epochs = int(dbutils.widgets.get("ctgan_epochs"))
ctgan_batch_size = int(dbutils.widgets.get("ctgan_batch_size"))
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
seed_val = int(dbutils.widgets.get("seed"))

source_table = f"{catalog}.{schema}.kaggle_fraud_detection"
output_table = f"{catalog}.{schema}.ctgan_fraud_{table_suffix}"

print(f"Source: {source_table}")
print(f"Output: {output_table}")
print(f"Target rows: {target_rows:,}")
print(f"Noise features: {n_noise_features}")
print(f"CTGAN epochs: {ctgan_epochs}, batch_size: {ctgan_batch_size}")

## Load Source Data

In [ ]:
import time
import pandas as pd
import numpy as np

load_start = time.time()

# Load fraud dataset
pdf = spark.table(source_table).toPandas()

# Drop high-cardinality string columns (same as autoresearch notebook)
cols_to_drop = [c for c in pdf.columns if pdf[c].dtype == 'object' and pdf[c].nunique() > 50]
pdf = pdf.drop(columns=cols_to_drop)
print(f"Dropped high-cardinality cols: {cols_to_drop}")

# Rename target for consistency
if "isFraud" in pdf.columns:
    pdf = pdf.rename(columns={"isFraud": "label"})

load_time = time.time() - load_start
print(f"Loaded {len(pdf):,} rows x {len(pdf.columns)} cols in {load_time:.1f}s")
print(f"Columns: {list(pdf.columns)}")
print(f"Label distribution:\n{pdf['label'].value_counts(normalize=True)}")

## Train CTGAN

In [ ]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

# Build metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(pdf)

# CTGAN synthesizer
print(f"Training CTGAN (epochs={ctgan_epochs}, batch_size={ctgan_batch_size})...")
train_start = time.time()

synthesizer = CTGANSynthesizer(
    metadata,
    epochs=ctgan_epochs,
    batch_size=ctgan_batch_size,
    verbose=True,
)
synthesizer.fit(pdf)

ctgan_train_time = time.time() - train_start
print(f"CTGAN training done in {ctgan_train_time:.1f}s ({ctgan_train_time/60:.1f} min)")

## Generate Synthetic Rows

In [ ]:
print(f"Generating {target_rows:,} synthetic rows...")
gen_start = time.time()

# Generate in chunks to avoid memory issues
CHUNK_SIZE = min(500_000, target_rows)
chunks = []
remaining = target_rows

while remaining > 0:
    chunk_n = min(CHUNK_SIZE, remaining)
    chunk = synthesizer.sample(num_rows=chunk_n)
    chunks.append(chunk)
    remaining -= chunk_n
    print(f"  Generated {target_rows - remaining:,}/{target_rows:,}")

synth_pdf = pd.concat(chunks, ignore_index=True)
gen_time = time.time() - gen_start
print(f"Generation done in {gen_time:.1f}s")
print(f"Shape: {synth_pdf.shape}")
print(f"Label distribution:\n{synth_pdf['label'].value_counts(normalize=True)}")

## Add Noise Features

The fraud dataset only has ~8 features. Add synthetic noise features
to reach the target feature count for benchmarking.

In [ ]:
if n_noise_features > 0:
    rng = np.random.default_rng(seed_val)
    print(f"Adding {n_noise_features} noise features...")

    # Mix of distributions for noise features
    for i in range(n_noise_features):
        dist_type = i % 4
        if dist_type == 0:  # gaussian
            synth_pdf[f"noise_f{i}"] = rng.standard_normal(len(synth_pdf)).astype(np.float32)
        elif dist_type == 1:  # uniform
            synth_pdf[f"noise_f{i}"] = rng.uniform(-3, 3, len(synth_pdf)).astype(np.float32)
        elif dist_type == 2:  # lognormal
            synth_pdf[f"noise_f{i}"] = rng.lognormal(0, 0.5, len(synth_pdf)).astype(np.float32)
        elif dist_type == 3:  # zero-inflated
            vals = rng.standard_normal(len(synth_pdf)).astype(np.float32)
            mask = rng.random(len(synth_pdf)) < 0.3
            vals[mask] = 0.0
            synth_pdf[f"noise_f{i}"] = vals

    print(f"Final shape: {synth_pdf.shape}")

## Write to Delta Table

In [ ]:
print(f"Writing to: {output_table}")
write_start = time.time()

# Convert to Spark DataFrame and write
sdf = spark.createDataFrame(synth_pdf)

n_partitions = max(8, target_rows // 2_000_000)
sdf = sdf.repartition(max(n_partitions, 8))

sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

write_time = time.time() - write_start
total_time = time.time() - load_start
print(f"Write: {write_time:.1f}s | Total: {total_time:.1f}s")

## Validate

In [ ]:
df_check = spark.table(output_table)
row_count = df_check.count()
print(f"Rows: {row_count:,} (expected {target_rows:,})")
print(f"Columns: {len(df_check.columns)}")

# Class distribution
label_counts = df_check.groupBy("label").count().orderBy("label").collect()
for row in label_counts:
    pct = row["count"] / row_count * 100
    print(f"  Label {row['label']}: {row['count']:,} ({pct:.2f}%)")

## Quick XGBoost Sanity Check

In [ ]:
if target_rows <= 100_000:
    import xgboost as xgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, average_precision_score

    check_pdf = df_check.toPandas()
    feature_cols = [c for c in check_pdf.columns if c != "label"]
    # Encode any remaining string columns
    for c in feature_cols:
        if check_pdf[c].dtype == 'object':
            check_pdf[c] = check_pdf[c].astype('category').cat.codes.astype(np.float32)

    X = check_pdf[sorted(feature_cols)].values.astype(np.float32)
    y = check_pdf["label"].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    neg, pos = np.sum(y_tr == 0), np.sum(y_tr == 1)
    spw = neg / max(pos, 1)

    model = xgb.XGBClassifier(
        tree_method="hist", n_estimators=200, max_depth=8,
        learning_rate=0.1, scale_pos_weight=spw, random_state=42, verbosity=0
    )
    model.fit(X_tr, y_tr)
    y_proba = model.predict_proba(X_te)[:, 1]
    auc_roc = roc_auc_score(y_te, y_proba)
    auc_pr = average_precision_score(y_te, y_proba)
    print(f"\nXGBoost sanity: AUC-ROC={auc_roc:.4f} | AUC-PR={auc_pr:.4f}")
else:
    print("Skipping sanity check (too large)")
    auc_roc, auc_pr = None, None

## Exit

In [ ]:
import json

result = {
    "status": "ok",
    "table": output_table,
    "source_table": source_table,
    "target_rows": target_rows,
    "actual_rows": row_count,
    "n_real_features": len(pdf.columns) - 1,
    "n_noise_features": n_noise_features,
    "total_features": len(df_check.columns) - 1,
    "ctgan_epochs": ctgan_epochs,
    "ctgan_train_time_sec": round(ctgan_train_time, 1),
    "total_time_sec": round(total_time, 1),
}
if auc_roc is not None:
    result["sanity_auc_roc"] = round(auc_roc, 4)
    result["sanity_auc_pr"] = round(auc_pr, 4)

result_json = json.dumps(result)
print(f"\nResult: {result_json}")
dbutils.notebook.exit(result_json)